In [50]:
import json
import pandas as pd

BATCH_SIZE = 50
IS_DEBUG = True
DEBUG_QUESTION_SIZE = 10
LLM_MODEL = "gpt-4.1"
DATASET_NAME = "hotpotQA"
KNOWLEDGE_GRAPH_PATH = 'outputs/knowledge_graphs'

## 1. Loading HotpotQA Dataset

In [51]:
train_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_train_v1.1.json'
test_hotpot_qa_path = r'C:\Users\Sudheera\Documents\Phd\LLMs_and_KGs\hotpotQA\hotpot_test_v1.1.json'

In [52]:
with open(train_hotpot_qa_path, 'r', encoding='utf-8') as f:
    train_hotpot_qa_json = json.load(f)

with open(test_hotpot_qa_path, 'r', encoding='utf-8') as f:
    test_hotpot_qa_json = json.load(f)

In [53]:
train_hotpot_qa_json[0]

{'supporting_facts': [["Arthur's Magazine", 0], ['First for Women', 0]],
 'level': 'medium',
 'question': "Which magazine was started first Arthur's Magazine or First for Women?",
 'context': [['Radio City (Indian radio station)',
   ["Radio City is India's first private FM radio station and was started on 3 July 2001.",
    ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
    ' It plays Hindi, English and regional songs.',
    ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.',
    ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.',
    ' The Radio station currently plays a mix of Hindi and Regional music.',
    ' Abraham Thomas is the CEO of the company.']]

In [54]:
test_hotpot_qa_json[0]

{'_id': '5a8b57f25542995d1e6f1371',
 'answer': 'yes',
 'question': 'Were Scott Derrickson and Ed Wood of the same nationality?',
 'supporting_facts': [['Scott Derrickson', 0], ['Ed Wood', 0]],
 'context': [['Adam Collis',
   ['Adam Collis is an American filmmaker and actor.',
    ' He attended the Duke University from 1986 to 1990 and the University of California, Los Angeles from 2007 to 2010.',
    ' He also studied cinema at the University of Southern California from 1991 to 1997.',
    ' Collis first work was the assistant director for the Scott Derrickson\'s short "Love in the Ruins" (1995).',
    ' In 1998, he played "Crankshaft" in Eric Koyanagi\'s "Hundred Percent".']],
  ['Ed Wood (film)',
   ['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
    " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lug

In [55]:
for item in train_hotpot_qa_json[0]['context']:
    print(f"Title: {item[0]}")
    print(f"Paragraph: {item[1:]}")
    print("-" * 50)

Title: Radio City (Indian radio station)
Paragraph: [["Radio City is India's first private FM radio station and was started on 3 July 2001.", ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).', ' It plays Hindi, English and regional songs.', ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.', ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.', ' The Radio station currently plays a mix of Hindi and Regional music.', ' Abraham Thomas is the CEO of the company.']]
--------------------------------------------------
Title: History of Albanian football
Paragraph: [['Football in Albania existed before the Albanian Football Federation (FSHF) was created.', " This was ev

In [56]:
train_batch_context = []
train_batch_question = []
train_batch_answer = []

if IS_DEBUG:
    train_hotpot_qa_json = train_hotpot_qa_json[:DEBUG_QUESTION_SIZE]
    test_hotpot_qa_json = test_hotpot_qa_json[:DEBUG_QUESTION_SIZE]

for i in range(0, len(train_hotpot_qa_json), BATCH_SIZE):
    batch = train_hotpot_qa_json[i:i + BATCH_SIZE]
    context = []
    question = []
    answer = []

    for item in batch:
        context_text = ""
        j = 1
        for ctx in item['context']:
            context_text += f"Title {j} : {ctx[0]} \nParagraph {j} : {''.join(ctx[1])}\n"
            j += 1
        context.append(context_text)
        question.append(item['question'])
        answer.append(item['answer'])

    train_batch_context.append(context)
    train_batch_question.append(question)
    train_batch_answer.append(answer)

In [57]:
if IS_DEBUG:
    batch_context = train_batch_context
    batch_question = train_batch_question
    batch_answer = train_batch_answer
else:
    batch_context = []
    batch_question = []
    batch_answer = []

In [58]:
for i in range(len(train_batch_context[0])):
    print(f"Batch {i + 1}:")
    print("Context:", train_batch_context[0][i])
    print("Question:", train_batch_question[0][i])
    print("Answer:", train_batch_answer[0][i])
    print("-" * 50)
    if i == 9:  # Limit to 3 batches for brevity
        break

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [59]:
del train_hotpot_qa_json
del test_hotpot_qa_json

## 2. Getting Triples from Context and Question

In [60]:
from dotenv import load_dotenv
import os

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

In [61]:
from langchain_openai import ChatOpenAI
from langchain_core.globals import set_debug, set_verbose, set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

set_debug(False)
set_verbose(False)
set_llm_cache(InMemoryCache())

llm_model = ChatOpenAI(model=LLM_MODEL, api_key=openai_api_key)

In [62]:
system_msg = """
<role>
    You are an expert in natural language understanding and knowledge graph construction. Your task is to extract structured relationship triples from a context set (Titles and Paragraphs) and to convert a provided question into a set of question triples, using simple variable names (such as x, y, z, i, j, k) for unknown entities or values to be inferred.
</role>

<behavior>
    <rule name="Entity Extraction">
        Identify and extract unique entities from Titles and Paragraphs. Use canonical names.
    </rule>
    <rule name="Relationship Extraction">
        Extract explicit or clear implicit relationships between entities as (subject, relation, object) triples.
    </rule>
    <rule name="Temporal and Comparative Relations">
        Use relations like "started in", "founded by", "wrote a song about", "is named after", etc., matching the logic and semantics in the context or question.
    </rule>
    <rule name="Do Not Use Question for Context Triples">
        When extracting triples from the context, **do not consider the question or its requirements**. Extract all possible triples from the context, regardless of the question.
    </rule>
    <rule name="Variable Placeholders for Unknowns">
        For each unknown or answer to be found in the question, use a simple English variable (x, y, z, i, j, k, etc.) in place of the answer within the triples. Each distinct unknown in the question gets a unique variable.
    </rule>
    <rule name="Question Triple Decomposition">
        For each question, decompose it into one or more triples, using variable placeholders for any unknowns to be inferred from the context.
        Examples:
        - "Which magazine was started first Arthur's Magazine or First for Women?"
           (Arthur's Magazine, started in, x)
           (First for Women, started in, y)
        - "The Oberoi family is part of a hotel company that has a head office in what city?"
           (Oberoi family, is part of, x)
           (x, has head office in, y)
    </rule>
    <rule name="Formatting and Output">
        Output must contain four sections, in order:
        1. ### CONTEXT_REASONING — a step-by-step reasoning of how context triples were extracted
        2. ### CONTEXT_TRIPLES — list all extracted context triples, each on its own line
        3. ### QUESTION_REASONING — a step-by-step reasoning of how the question was decomposed into question triples
        4. ### QUESTIONS_TRIPLES — list the question triples, each on its own line
        Do not add explanations, numbering, or extra text.
    </rule>
</behavior>

<format>
1. Carefully read the <context> section.
2. Before extracting triples, explain step by step how you identify and select the entities and relationships in the context. Print this heading:
   ### CONTEXT_REASONING
   Then write your reasoning in clear, concise steps. Extract as many triples as you can from the context. **Do not focus on the question yet.**
3. After reasoning, extract and print context triples. Print this heading:
   ### CONTEXT_TRIPLES
   List each context triple on a new line. Extract as many triples as you can from the context. **Do not focus on the question yet.**
4. Carefully read the <question> section.
5. Before extracting question triples, explain step by step how you break down the question and assign variable placeholders. Print this heading:
   ### QUESTION_REASONING
   Then write your reasoning in clear, concise steps.
6. After reasoning, print the question triples. Print this heading:
   ### QUESTIONS_TRIPLES
   List each question triple on a new line, using variable placeholders (x, y, z, etc.) for unknowns.
7. After the last question triple, end with no extra text or parentheses.
</format>
"""

human_msg = """
<context>
{context}
</context>

<question>
{question}
</question>
"""

prompt = ChatPromptTemplate([("system", system_msg), ("human", human_msg)])

In [63]:
chain = prompt | llm_model | StrOutputParser()

In [64]:
batch_input_list = []
main_df = pd.DataFrame()

for i in range(len(batch_context)):
    batch_num_list = [i] * len(batch_context[i])
    question_num_list = list(range(1, len(batch_context[i]) + 1))
    context_list = []
    question_list = []
    answers_list = []

    batch_input_item = []
    j = 0
    for batch in range(len(batch_context[i])):
        context = batch_context[i][batch]
        batch_input_item.append({
            "context": context,
            "question": batch_question[i][j],
        })
        context_list.append(context)
        question_list.append(batch_question[i][j])
        answers_list.append(batch_answer[i][j])
        j += 1
    batch_input_list.append(batch_input_item)
    df = pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "context": context_list,
        "question": question_list,
        "answer": answers_list
    })
    main_df = pd.concat([main_df, df], ignore_index=True)

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1}:")
    for item in batch:
        print("Context:", item['context'])
        print("Question:", item['question'])
        print("-" * 50)

Batch 1:
Context: Title 1 : Radio City (Indian radio station) 
Paragraph 1 : Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title 2 : History of Albanian football 
Paragraph 2 : Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tou

In [65]:
main_df

,batch_num,question_num,context,question,answer
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long


In [16]:
graphs_str_batch = []

for i, batch in enumerate(batch_input_list):
    print(f"Batch {i + 1} ... processing {len(batch)} items")
    graphs_str_list = chain.batch(batch)
    graphs_str_batch.append(graphs_str_list)

Batch 1 ... processing 10 items


In [17]:
for i, graphs_str_list in enumerate(graphs_str_batch):
    print(f"Batch {i + 1} ... processed {len(graphs_str_list)} items")
    for j, graph_str in enumerate(graphs_str_list):
        print(f"Item {j + 1}:")
        print(graph_str)
        print("-" * 50)

Batch 1 ... processed 10 items
Item 1:
### CONTEXT_REASONING
I begin by reading each title and paragraph, identifying key entities such as organizations, people, songs, places, dates, and relationships like "founded by," "started in," "played," "merged into," etc. For each magazine mentioned, like "Arthur's Magazine" and "First for Women," I look for relevant facts: who started them, when, where, and other notable events. For other contexts, I extract relationships such as "played in," "started as," "published in," "founded by," "merged into," "defeated," etc. I ensure all unique entities are covered and explicit and clear implicit relationships are captured in triple format, always focusing only on the context regardless of the question.

### CONTEXT_TRIPLES
(Radio City, is, India's first private FM radio station)
(Radio City, was started on, 3 July 2001)
(Radio City, broadcasts on, 91.1 megahertz)
(Radio City, earlier broadcasted on, 91.0 megahertz)
(Radio City, broadcasts from, Mumb

In [44]:
graphs_str_df = pd.DataFrame({
    "batch_num": [],
    "question_num": [],
    "graphs_str": []
})

for i, graphs_str_list in enumerate(graphs_str_batch):
    batch_num_list = [i] * len(graphs_str_list)
    question_num_list = list(range(1, len(graphs_str_list) + 1))
    graphs_str_df = pd.concat([graphs_str_df, pd.DataFrame({
        "batch_num": batch_num_list,
        "question_num": question_num_list,
        "graphs_str": graphs_str_list
    })], ignore_index=True)

del graphs_str_batch
del graphs_str_list
del batch_input_list
del batch_context
del batch_question
del batch_answer

graphs_str_df

,batch_num,question_num,graphs_str
0,0.0,1.0,### CONTEXT_REASONING\nI begin by reading each...
1,0.0,2.0,### CONTEXT_REASONING\nI first identify unique...
2,0.0,3.0,### CONTEXT_REASONING\nTo extract context trip...
3,0.0,4.0,### CONTEXT_REASONING\nI start by reading each...
4,0.0,5.0,### CONTEXT_REASONING\nTo extract context trip...
5,0.0,6.0,"### CONTEXT_REASONING\nFirst, I identify named..."
6,0.0,7.0,### CONTEXT_REASONING\nTo extract context trip...
7,0.0,8.0,"### CONTEXT_REASONING\nFirst, I identify indiv..."
8,0.0,9.0,"### CONTEXT_REASONING\n- First, I scan all par..."
9,0.0,10.0,"### CONTEXT_REASONING\nFirst, I identify the k..."


## 3. Parsing the Output and Building KGs

In [66]:
import re
from ast import literal_eval

def sanitize_str(name):
    """
    Sanitize a name by removing extra spaces, apostrophes, and ensuring proper formatting.
    """
    # Replace apostrophes with empty string
    name = name.replace("'s", "s")
    name = name.replace("'", "")

    # Replace spaces with underscores
    name = name.replace(" ", "_")

    # Remove any other problematic characters
    name = re.sub(r'[^\w\-_]', '', name)

    # Prefix if starts with digit
    if re.match(r"^\d", name):
        name = f"n{name}"

    # If empty string, return something safe
    if not name:
        name = "unknown"

    return name

def sanitize_entities_and_relations(triple):
    """
    Sanitize entities and relations in a triple by removing extra spaces and ensuring proper formatting.
    """
    return (
        sanitize_str(triple[0].strip()),  # Subject
        sanitize_str(triple[1].strip()),  # Relation
        sanitize_str(triple[2].strip())   # Object
    )


def parse_custom_triples(triples_str):
    triples = []
    for line in triples_str.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        # Remove enclosing parentheses
        if line.startswith('(') and line.endswith(')'):
            line = line[1:-1]
        # Now, split ONLY on the first two commas
        parts = []
        remaining = line
        for _ in range(2):
            # Find the first comma
            idx = remaining.find(',')
            if idx == -1:
                break
            parts.append(remaining[:idx].strip())
            remaining = remaining[idx+1:].strip()
        parts.append(remaining)
        if len(parts) == 3:
            triples.append(sanitize_entities_and_relations(tuple(parts)))
    return triples

def extract_sections(text):
    # Regex patterns for section headers
    context_reasoning_pat = r'### CONTEXT_REASONING\s*(.*?)\s*### CONTEXT_TRIPLES'
    context_triples_pat = r'### CONTEXT_TRIPLES\s*(.*?)\s*### QUESTION_REASONING'
    question_reasoning_pat = r'### QUESTION_REASONING\s*(.*?)\s*### QUESTIONS_TRIPLES'
    questions_triples_pat = r'### QUESTIONS_TRIPLES\s*(.*)$'

    # Extract sections using regex
    context_reasoning = re.search(context_reasoning_pat, text, re.DOTALL)
    context_triples = re.search(context_triples_pat, text, re.DOTALL)
    question_reasoning = re.search(question_reasoning_pat, text, re.DOTALL)
    questions_triples = re.search(questions_triples_pat, text, re.DOTALL)

    # Clean reasoning sections
    context_reasoning_str = context_reasoning.group(1).strip() if context_reasoning else ""
    question_reasoning_str = question_reasoning.group(1).strip() if question_reasoning else ""

    # Use the custom parser for triples
    context_triples_list = parse_custom_triples(context_triples.group(1)) if context_triples else []
    questions_triples_list = parse_custom_triples(questions_triples.group(1)) if questions_triples else []

    return context_reasoning_str, question_reasoning_str, context_triples_list, questions_triples_list

def extract_to_columns(row):
    context_reasoning, question_reasoning, context_triples, question_triples = extract_sections(row['graphs_str'])
    return pd.Series([context_reasoning, question_reasoning, context_triples, question_triples],
                     index=['context_reasoning', 'question_reasoning', 'context_triples', 'question_triples'])

In [67]:
graphs_str_df[['context_reasoning', 'question_reasoning', 'context_triples', 'question_triples']] = graphs_str_df.apply(extract_to_columns, axis=1)
graphs_str_df

,batch_num,question_num,graphs_str,context_reasoning,question_reasoning,context_triples,question_triples
0,0.0,1.0,### CONTEXT_REASONING\nI begin by reading each...,"I begin by reading each title and paragraph, i...",The question asks which magazine was started f...,"[(Radio_City, is, Indias_first_private_FM_radi...","[(Arthurs_Magazine, was_started_on, x), (First..."
1,0.0,2.0,### CONTEXT_REASONING\nI first identify unique...,"I first identify unique entities: ""Ritz-Carlto...",The question asks for the city in which the he...,"[(Ritz-Carlton_Jakarta, is_a, hotel), (Ritz-Ca...","[(Oberoi_family, is_part_of, x), (x, has_head_..."
2,0.0,3.0,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I first identify a...",The question asks:\n- Allie Goertz wrote a son...,"[(Lisa_Simpson, is_a_character_in, The_Simpson...","[(Matt_Groening, named_Milhouse_Van_Houten_aft..."
3,0.0,4.0,### CONTEXT_REASONING\nI start by reading each...,"I start by reading each title and paragraph, i...","The question asks for the nationality of ""Jame...","[(Moloch_or, This_Gentile_World, is_a_semi-aut...","[(Ewan_MacColl, was_married_to, Peggy_Seeger),..."
4,0.0,5.0,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I first identify t...",The question asks for a chemical in which cadm...,"[(Cadmium_chloride, is_a_compound_of, cadmium)...","[(Cadmium_chloride, is_slightly_soluble_in, x)..."
5,0.0,6.0,"### CONTEXT_REASONING\nFirst, I identify named...","First, I identify named entities within the co...",The question asks which tennis player won more...,"[(Li_Na, is_a, Chinese_professional_tennis_pla...","[(Henri_Leconte, won, x_Grand_Slam_titles), (J..."
6,0.0,7.0,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I identify all uni...",The question asks for the genus of moth in the...,"[(India, is_officially_known_as, Republic_of_I...","[(x, is_a, genus_of_moth), (x, is_found_in, In..."
7,0.0,8.0,"### CONTEXT_REASONING\nFirst, I identify indiv...","First, I identify individual entities—people, ...","I read the question, which asks for the name o...","[(Verano_de_Escándalo_1998, is_a, professional...","[(x, was_once_considered, best_kickboxer_in_th..."
8,0.0,9.0,"### CONTEXT_REASONING\n- First, I scan all par...","- First, I scan all paragraphs for named entit...",- The question asks about the Dutch-Belgian te...,"[(House_of_Anubis, is_a, mystery_television_se...","[(Het_Huis_Anubis, first_aired_in, x)]"
9,0.0,10.0,"### CONTEXT_REASONING\nFirst, I identify the k...","First, I identify the key entities mentioned i...",The question asks for the length of the track ...,"[(Mount_Panorama_Circuit, is_a, motor_racing_t...","[(n2013_Liqui_Moly_Bathurst_12_Hour, staged_at..."


In [68]:
main_df = main_df.merge(graphs_str_df, on=['batch_num', 'question_num'], how='left')
# del graphs_str_df
main_df

,batch_num,question_num,context,question,answer,graphs_str,context_reasoning,question_reasoning,context_triples,question_triples
0,0,1,Title 1 : Radio City (Indian radio station) \n...,Which magazine was started first Arthur's Maga...,Arthur's Magazine,### CONTEXT_REASONING\nI begin by reading each...,"I begin by reading each title and paragraph, i...",The question asks which magazine was started f...,"[(Radio_City, is, Indias_first_private_FM_radi...","[(Arthurs_Magazine, was_started_on, x), (First..."
1,0,2,Title 1 : Ritz-Carlton Jakarta \nParagraph 1 :...,The Oberoi family is part of a hotel company t...,Delhi,### CONTEXT_REASONING\nI first identify unique...,"I first identify unique entities: ""Ritz-Carlto...",The question asks for the city in which the he...,"[(Ritz-Carlton_Jakarta, is_a, hotel), (Ritz-Ca...","[(Oberoi_family, is_part_of, x), (x, has_head_..."
2,0,3,Title 1 : Lisa Simpson \nParagraph 1 : Lisa Ma...,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I first identify a...",The question asks:\n- Allie Goertz wrote a son...,"[(Lisa_Simpson, is_a_character_in, The_Simpson...","[(Matt_Groening, named_Milhouse_Van_Houten_aft..."
3,0,4,"Title 1 : Moloch: or, This Gentile World \nPar...",What nationality was James Henry Miller's wife?,American,### CONTEXT_REASONING\nI start by reading each...,"I start by reading each title and paragraph, i...","The question asks for the nationality of ""Jame...","[(Moloch_or, This_Gentile_World, is_a_semi-aut...","[(Ewan_MacColl, was_married_to, Peggy_Seeger),..."
4,0,5,Title 1 : Cadmium chloride \nParagraph 1 : Cad...,Cadmium Chloride is slightly soluble in this c...,alcohol,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I first identify t...",The question asks for a chemical in which cadm...,"[(Cadmium_chloride, is_a_compound_of, cadmium)...","[(Cadmium_chloride, is_slightly_soluble_in, x)..."
5,0,6,Title 1 : Li Na \nParagraph 1 : Li Na (; ; bor...,Which tennis player won more Grand Slam titles...,Jonathan Stark,"### CONTEXT_REASONING\nFirst, I identify named...","First, I identify named entities within the co...",The question asks which tennis player won more...,"[(Li_Na, is_a, Chinese_professional_tennis_pla...","[(Henri_Leconte, won, x_Grand_Slam_titles), (J..."
6,0,7,"Title 1 : India \nParagraph 1 : India, officia...",Which genus of moth in the world's seventh-lar...,Crambidae,### CONTEXT_REASONING\nTo extract context trip...,"To extract context triples, I identify all uni...",The question asks for the genus of moth in the...,"[(India, is_officially_known_as, Republic_of_I...","[(x, is_a, genus_of_moth), (x, is_found_in, In..."
7,0,8,Title 1 : Verano de Escándalo (1998) \nParagra...,Who was once considered the best kick boxer in...,Badr Hari,"### CONTEXT_REASONING\nFirst, I identify indiv...","First, I identify individual entities—people, ...","I read the question, which asks for the name o...","[(Verano_de_Escándalo_1998, is_a, professional...","[(x, was_once_considered, best_kickboxer_in_th..."
8,0,9,Title 1 : House of Anubis \nParagraph 1 : Hous...,"The Dutch-Belgian television series that ""Hous...",2006,"### CONTEXT_REASONING\n- First, I scan all par...","- First, I scan all paragraphs for named entit...",- The question asks about the Dutch-Belgian te...,"[(House_of_Anubis, is_a, mystery_television_se...","[(Het_Huis_Anubis, first_aired_in, x)]"
9,0,10,Title 1 : Mount Panorama Circuit \nParagraph 1...,What is the length of the track where the 2013...,6.213 km long,"### CONTEXT_REASONING\nFirst, I identify the k...","First, I identify the key entities mentioned i...",The question asks for the length of the track ...,"[(Mount_Panorama_Circuit, is_a, motor_racing_t...","[(n2013_Liqui_Moly_Bathurst_12_Hour, staged_at..."


In [69]:
# Loop through each row. Save the context triples as OWL KG files
from rdflib import Graph, Namespace, RDF, RDFS, OWL, URIRef
import os

def save_kg_from_triples(triples, batch_num, question_num):
    g = Graph()
    EX = Namespace("http://example.org/")

    # Add triples to the graph
    for subject, relation, obj in triples:
        subject_uri = URIRef(EX[subject])
        object_uri = URIRef(EX[obj])
        g.add((subject_uri, URIRef(EX[relation]), object_uri))

    # Define the file path
    file_path = f"{KNOWLEDGE_GRAPH_PATH}/{DATASET_NAME}_ontology_b{batch_num}_q{question_num}.rdf"

    # Save the graph in OWL format
    g.serialize(destination=file_path, format='xml')
    print(f"Saved KG for batch {batch_num}, question {question_num} to {file_path}")

for index, row in main_df.iterrows():
    batch_num = row['batch_num']
    question_num = row['question_num']
    context_triples = row['context_triples']

    if context_triples:  # Only save if there are context triples
        save_kg_from_triples(context_triples, batch_num, question_num)

Saved KG for batch 0, question 1 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q1.rdf
Saved KG for batch 0, question 2 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q2.rdf
Saved KG for batch 0, question 3 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q3.rdf
Saved KG for batch 0, question 4 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q4.rdf
Saved KG for batch 0, question 5 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q5.rdf
Saved KG for batch 0, question 6 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q6.rdf
Saved KG for batch 0, question 7 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q7.rdf
Saved KG for batch 0, question 8 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q8.rdf
Saved KG for batch 0, question 9 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q9.rdf
Saved KG for batch 0, question 10 to outputs/knowledge_graphs/hotpotQA_ontology_b0_q10.rdf
